# Weather Bronze Layer — Auto Loader Ingestion

Ingests JSON files from `workspace.landing.weather_landing` volume using **Auto Loader** (`cloudFiles`) into `workspace.bronze.weather_readings`.

Uses `availableNow=True` trigger (serverless-compatible) — processes all new files then stops. Re-run to pick up new data.

In [0]:
from pyspark.sql.functions import *

SOURCE_PATH = "/Volumes/workspace/landing/weather_landing"
SCHEMA_LOCATION = "/Volumes/workspace/landing/schemas/weather/bronze"
CHECKPOINT_LOCATION = "/Volumes/workspace/landing/checkpoints/weather/bronze"
BRONZE_TABLE = "workspace.bronze.weather_readings"

spark.sql("CREATE VOLUME IF NOT EXISTS workspace.landing.weather_landing")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")
# Drop table only if it has zero columns (empty table from initial CREATE TABLE)
if spark.catalog.tableExists(BRONZE_TABLE):
    cols = [r.col_name for r in spark.sql(f"DESCRIBE TABLE {BRONZE_TABLE}").collect() if r.col_name and not r.col_name.startswith("#")]
    if len(cols) == 0:
        spark.sql(f"DROP TABLE {BRONZE_TABLE}")
        print(f"Dropped empty {BRONZE_TABLE}")

In [0]:
stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(SOURCE_PATH))

query = (stream.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT_LOCATION)
    .trigger(availableNow=True)
    .toTable(BRONZE_TABLE))

query.awaitTermination()
print("✅ Bronze stream completed — processed all available files")

In [0]:
row_count = spark.table(BRONZE_TABLE).count()
print(f"📊 Bronze table {BRONZE_TABLE}: {row_count} rows")
spark.table(BRONZE_TABLE).display()